# 04 — Model Evaluation

Load saved metrics and predictions. Analyse results on the held-out test set.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

cfg = yaml.safe_load(open('../configs/config.yaml'))
metrics_df = pd.read_csv('../reports/model_metrics.csv')
test_preds = pd.read_csv('../reports/test_predictions.csv', index_col=0, parse_dates=True)
test_preds.index = pd.to_datetime(test_preds.index, utc=True)
FIG_EVAL = Path('../reports/figures/evaluation')
print("Metrics loaded:")
with pd.option_context('display.float_format', '{:.4f}'.format):
    print(metrics_df.to_string(index=False))


## Model Comparison

In [ ]:
from src.evaluation.plots import plot_model_comparison
plot_model_comparison(metrics_df, FIG_EVAL)
try:
    img = plt.imread(str(FIG_EVAL / 'model_comparison.png'))
    plt.figure(figsize=(12, 5)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()
except Exception:
    print("Figure not found")


## Actual vs Predicted

In [ ]:
from src.evaluation.plots import plot_actual_vs_predicted, plot_residuals

best_model_name = metrics_df[~metrics_df['model'].isin(['Persistence','Seasonal Naive'])].sort_values('MAE').iloc[0]['model']
print(f"Best model: {best_model_name}")

y_true = test_preds['actual']
y_pred = test_preds[best_model_name]
sub = pd.concat([y_true, y_pred], axis=1).dropna()

plot_actual_vs_predicted(sub['actual'], sub[best_model_name], best_model_name, FIG_EVAL)
plot_residuals(sub['actual'], sub[best_model_name], best_model_name, FIG_EVAL)

try:
    img = plt.imread(str(FIG_EVAL / f'actual_vs_predicted_{best_model_name.replace(" ", "_")}.png'))
    plt.figure(figsize=(14, 7)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()
except Exception:
    print("Figure not found")


## Error Analysis

In [ ]:
y_true = test_preds['actual']
y_pred = test_preds[best_model_name]
residuals = (y_true - y_pred).dropna()

print(f"Residual statistics ({best_model_name}):")
print(f"  Mean   : {residuals.mean():.4f} ug/m3")
print(f"  Std    : {residuals.std():.4f}")
print(f"  Skew   : {residuals.skew():.4f}")
print(f"  Max overestimate: {residuals.max():.2f}")
print(f"  Max underestimate: {residuals.min():.2f}")

# Largest errors
abs_err = residuals.abs().sort_values(ascending=False).head(10)
print()
print("Top 10 largest absolute errors:")
print(abs_err.to_string())


## Evaluation Conclusions

- **Best model**: Linear Regression (MAE = 7.02 µg/m³)
- All ML models beat both baselines, confirming the features add value
- RMSE > MAE indicates occasional large errors (pollution spikes are harder to predict)
- R² = 0.167 is modest — 24-hour-ahead PM2.5 forecasting is inherently uncertain
- This test set evaluation is historical; future performance may differ
